# Playlist Matcher Test Notebook

This notebook creates a mock music library with the first 10 songs from Favourites.m3u8 and tests the playlist_matcher.py script.

## Setup

First, we'll install the required dependencies and import necessary modules.

In [1]:
# Install mutagen if not already installed
!pip install mutagen

In [14]:
import os
import shutil
from pathlib import Path
from mutagen.flac import FLAC
import re
import struct
from importlib import reload

import playlist_matcher as pm

## Helper Functions

Functions to sanitize filenames and handle special characters.

In [3]:
def sanitize_filename(name):
    """Sanitize a string for use in filenames by replacing problematic characters"""
    # Replace forward slash with unicode division slash or underscore
    # This preserves the visual appearance while being filesystem-safe
    name = name.replace('/', '∕')  # Unicode division slash U+2215
    # Could also use: name = name.replace('/', '_')
    
    # Replace other problematic characters
    replacements = {
        '\\': '∖',  # Backslash
        ':': '∶',   # Colon  
        '*': '∗',   # Asterisk
        '?': '？',  # Question mark
        '"': '＂',  # Quote
        '<': '＜',  # Less than
        '>': '＞',  # Greater than
        '|': '｜',  # Pipe
    }
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return name

def sanitize_path_component(name):
    """Sanitize directory/album names"""
    return sanitize_filename(name)

## Parse First 10 Entries from Favourites.m3u8

We'll read the playlist and extract metadata for the first 10 songs.

In [5]:
def parse_playlist_entries(playlist_path, num_entries=10):
    """Parse first N entries from playlist"""
    entries = []
    
    with open(playlist_path, 'r', encoding='utf-8-sig') as f:
        lines = [line.rstrip('\n\r') for line in f.readlines()]
    
    i = 0
    count = 0
    while i < len(lines) and count < num_entries:
        line = lines[i]
        
        if line.startswith('#EXTINF:'):
            if i + 1 < len(lines):
                extinf_line = line
                path_line = lines[i + 1]
                
                # Parse EXTINF: #EXTINF:duration,Artist - Title
                extinf_match = re.match(r'#EXTINF:(\d+),(.+)', extinf_line)
                if extinf_match:
                    duration = extinf_match.group(1)
                    artist_title = extinf_match.group(2)
                    
                    # Split artist and title
                    if ' - ' in artist_title:
                        artist, title = artist_title.split(' - ', 1)
                    else:
                        artist = ""
                        title = artist_title
                    
                    # Parse path: ..\Artist(s)\Album\CD# - Track# - Artist(s) - Title - Album.ext
                    clean_path = path_line.replace('..\\', '').replace('../', '').replace('\\', '/')
                    path_parts = clean_path.split('/')
                    
                    playlist_artist = path_parts[0] if len(path_parts) > 0 else artist
                    album = path_parts[1] if len(path_parts) > 1 else "Unknown Album"
                    filename = path_parts[2] if len(path_parts) > 2 else "unknown.flac"
                    
                    # Parse filename for additional metadata
                    # Format: CD# - Track# - Artist(s) - Title - Album.ext
                    # Use ' - ' (space-dash-space) as delimiter to allow '-' in names
                    filename_match = re.match(r'^(\d+) - (\d+) - (.+?) - (.+?) - (.+?)\.(\w+)$', filename)
                    
                    if filename_match:
                        disc = filename_match.group(1)
                        track = filename_match.group(2)
                        file_artist = filename_match.group(3)
                        file_title = filename_match.group(4)
                        file_album = filename_match.group(5)
                        ext = filename_match.group(6)
                    else:
                        disc = "1"
                        track = str(count + 1)
                        file_artist = artist
                        file_title = title
                        file_album = album
                        ext = "flac"
                    
                    entries.append({
                        'artist': artist.strip(),
                        'title': title.strip(),
                        'album': album.strip(),
                        'playlist_artist': playlist_artist.strip(),
                        'file_artist': file_artist.strip(),
                        'file_title': file_title.strip(),
                        'file_album': file_album.strip(),
                        'disc': disc,
                        'track': track,
                        'ext': ext,
                        'duration': duration,
                        'original_path': path_line
                    })
                    count += 1
                
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    return entries

# Parse first 10 entries
entries = parse_playlist_entries('../Favourites.m3u8', 10)

print(f"Parsed {len(entries)} entries:\n")
for i, entry in enumerate(entries, 1):
    print(f"{i}. {entry['artist']} - {entry['title']}")
    print(f"   Album: {entry['album']}")
    print(f"   Track: {entry['disc']}-{entry['track']}")
    print(f"   File title: {entry['file_title']}")
    print()

Parsed 10 entries:

1. The Offspring - (Can't Get My) Head Around You
   Album: Splinter
   Track: 1-164
   File title: (Can't Get My) Head Around You

2. Santana - (Da Le) Yaleo
   Album: Supernatural (Remastered)
   Track: 1-208
   File title: (Da Le) Yaleo

3. Jamiroquai - (Don't) Give Hate a Chance
   Album: Dynamite
   Track: 1-247
   File title: (Don't) Give Hate a Chance

4. The Rolling Stones - (I Can't Get No) Satisfaction - Mono
   Album: Out Of Our Heads
   Track: 1-458
   File title: (I Can't Get No) Satisfaction

5. JAY-Z, Beyoncé - 03' Bonnie & Clyde
   Album: The Blueprint 2 The Gift & The Curse
   Track: 1-1
   File title: 03' Bonnie & Clyde

6. Die Ärzte - 1/2 Lovesong
   Album: 13
   Track: 1-2
   File title: 12 Lovesong

7. Ciara, Missy Elliott - 1, 2 Step (feat. Missy Elliott)
   Album: Goodies
   Track: 1-3
   File title: 1, 2 Step (feat. Missy Elliott)

8. Gorillaz - 19-2000 - Soulchild Remix
   Album: Gorillaz
   Track: 1-4
   File title: 19-2000

9. A Tribe Call

## Create Mock Library Using Template

Now we'll copy the template FLAC file and modify its metadata for each song.
We sanitize filenames to handle special characters like '/' safely.

In [6]:
def create_mock_library(entries, template_path='example.flac', base_dir='test_music_library'):
    """Create mock music library by copying template and modifying metadata"""
    
    if not os.path.exists(template_path):
        print(f"Error: Template file {template_path} not found!")
        return []
    
    # Clean up existing directory
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    
    os.makedirs(base_dir)
    
    created_files = []
    
    for entry in entries:
        # Use artist as album artist for the directory structure
        album_artist = entry['artist']
        album = entry['album']
        
        # Sanitize directory names
        safe_artist = sanitize_path_component(album_artist)
        safe_album = sanitize_path_component(album)
        
        # Create directory structure
        album_dir = Path(base_dir) / safe_artist / safe_album
        album_dir.mkdir(parents=True, exist_ok=True)
        
        # Create filename in library format with sanitized components
        # Format: CD# - Track# - Title - Artist(s) - Album.ext
        safe_title = sanitize_filename(entry['title'])
        safe_artist_name = sanitize_filename(entry['artist'])
        safe_album_name = sanitize_filename(entry['album'])
        
        filename = f"{entry['disc']} - {entry['track']} - {safe_title} - {safe_artist_name} - {safe_album_name}.{entry['ext']}"
        file_path = album_dir / filename
        
        try:
            # Copy template file
            shutil.copy2(template_path, file_path)
            
            # Modify metadata - use ORIGINAL unsanitized values for metadata tags
            audio = FLAC(str(file_path))
            audio['title'] = entry['title']  # Original with /
            audio['artist'] = entry['artist']
            audio['album'] = entry['album']
            audio['albumartist'] = album_artist
            audio['tracknumber'] = entry['track']
            audio['discnumber'] = entry['disc']
            audio.save()
            
            # Verify
            audio = FLAC(str(file_path))
            print(f"✓ Created: {file_path.relative_to(base_dir)}")
            print(f"  Metadata: '{audio.get('title', [''])[0]}' by '{audio.get('artist', [''])[0]}'")
            
            created_files.append(str(file_path))
            
        except Exception as e:
            print(f"✗ Failed to create: {file_path.relative_to(base_dir) if file_path.exists() else filename}")
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
    
    return created_files

# Create the mock library
print("Creating mock music library from template...\n")
created_files = create_mock_library(entries)
print(f"\nCreated {len(created_files)} files in test_music_library/")

Creating mock music library from template...

✓ Created: The Offspring\Splinter\1 - 164 - (Can't Get My) Head Around You - The Offspring - Splinter.flac
  Metadata: '(Can't Get My) Head Around You' by 'The Offspring'
✓ Created: Santana\Supernatural (Remastered)\1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
  Metadata: '(Da Le) Yaleo' by 'Santana'
✓ Created: Jamiroquai\Dynamite\1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
  Metadata: '(Don't) Give Hate a Chance' by 'Jamiroquai'
✓ Created: The Rolling Stones\Out Of Our Heads\1 - 458 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Out Of Our Heads.flac
  Metadata: '(I Can't Get No) Satisfaction - Mono' by 'The Rolling Stones'
✓ Created: JAY-Z, Beyoncé\The Blueprint 2 The Gift & The Curse\1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
  Metadata: '03' Bonnie & Clyde' by 'JAY-Z, Beyoncé'
✓ Created: Die Ärzte\13\1 - 2 - 1∕2 Lovesong - Die Ärzte -

## Create Test Playlist

Create a small test playlist with just these 10 songs.

In [7]:
def create_test_playlist(entries, output_path='test_playlist.m3u8'):
    """Create a test playlist with the parsed entries"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('#EXTM3U\n')
        for entry in entries:
            f.write(f"#EXTINF:{entry['duration']},{entry['artist']} - {entry['title']}\n")
            f.write(f"{entry['original_path']}\n")
    
    print(f"Created test playlist: {output_path}")

create_test_playlist(entries)

Created test playlist: test_playlist.m3u8


## Test the Playlist Matcher Script

Now let's run the playlist matcher script on our test data.

In [9]:
# Run the playlist matcher
!python playlist_matcher.py \
    --playlist test_playlist.m3u8 \
    --music-dir test_music_library \
    --output test_output.m3u8 \
    --log test_unmatched.log \
    --format artist_album

2026-01-23 20:41:05,688 - INFO - Using playlist path format: artist_album
2026-01-23 20:41:05,688 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext
2026-01-23 20:41:05,688 - INFO - Step 1: Building library cache
2026-01-23 20:41:05,688 - INFO - Scanning music library: test_music_library
2026-01-23 20:41:05,688 - INFO - Processing album artist: 2Pac, Snoop Dogg
2026-01-23 20:41:05,706 - INFO - Processing album artist: A Tribe Called Quest
2026-01-23 20:41:05,706 - INFO - Processing album artist: Ciara, Missy Elliott
2026-01-23 20:41:05,930 - INFO - Processing album artist: Die Ärzte
2026-01-23 20:41:05,937 - INFO - Processing album artist: Gorillaz
2026-01-23 20:41:05,938 - INFO - Processing album artist: Jamiroquai
2026-01-23 20:41:05,939 - INFO - Processing album artist: JAY-Z, Beyoncé
2026-01-23 20:41:05,939 - INFO - Processing album artist: Santana
2026-01-23 20:41:05,940 - INFO - Processing album artist: The Offspring
2026-01-23 20:41:05,940 

## Verify Results

Let's check the output playlist and unmatched log.

In [ ]:
print("=" * 80)
print("OUTPUT PLAYLIST (test_output.m3u8)")
print("=" * 80)
if os.path.exists('test_output.m3u8'):
    with open('test_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("UNMATCHED LOG (test_unmatched.log)")
print("=" * 80)
if os.path.exists('test_unmatched.log'):
    with open('test_unmatched.log', 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("File not found!")

## Verify File Structure

Let's verify the created library structure.

In [ ]:
print("Test Music Library Structure:\n")
for root, dirs, files in os.walk('test_music_library'):
    level = root.replace('test_music_library', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        print(f"{subindent}{file}")

In [12]:
# Run the playlist matcher
!python playlist_matcher.py \
    --playlist "E:\Music\_playlists\Favourites.m3u8" \
    --music-dir "E:\Music" \
    --output "new_favs.m3u8" \
    --log "unmatched.log" \
    --format artist_album

2026-01-23 20:44:07,268 - INFO - Using playlist path format: artist_album
2026-01-23 20:44:07,268 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext
2026-01-23 20:44:07,268 - INFO - Step 1: Building library cache
2026-01-23 20:44:07,268 - INFO - Scanning music library: E:\Music
2026-01-23 20:44:07,274 - INFO - Processing album artist: 19 Naughty III (US Release)
2026-01-23 20:44:07,274 - INFO - Processing album artist: 2 Brothers On The 4th Floor
2026-01-23 20:44:07,286 - INFO - Processing album artist: 2 Unlimited
2026-01-23 20:44:07,298 - INFO - Processing album artist: 2Pac
2026-01-23 20:44:07,319 - INFO - Cached 100 files...
2026-01-23 20:44:07,330 - INFO - Processing album artist: 3 Steps Ahead
2026-01-23 20:44:07,330 - INFO - Processing album artist: 4am Kru
2026-01-23 20:44:07,331 - INFO - Processing album artist: 50 Cent
2026-01-23 20:44:07,354 - INFO - Cached 200 files...
2026-01-23 20:44:07,367 - INFO - Processing album artist: 6 Pack (E

In [19]:
reload(pm)
base = pm.PlaylistMatcher(
    playlist_path="E:\\Music\\_playlists\\How Hard Can It Be.m3u8",
    music_dir="E:\\Music",
    output_path="E:\\Music\\_playlists\\How Hard Can It Be - Matched.m3u8",
    log_path="E:\\Music\\_playlists\\unmatched.log",
    path_format="artist_album"
)

2026-01-23 22:21:47,403 - INFO - Using playlist path format: artist_album
2026-01-23 22:21:47,403 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext


In [20]:
base.build_library_cache()

2026-01-23 22:22:17,183 - INFO - Step 1: Building library cache
2026-01-23 22:22:17,184 - INFO - Scanning music library: E:\Music
2026-01-23 22:22:17,190 - INFO - Processing album artist: 19 Naughty III (US Release)
2026-01-23 22:22:17,191 - INFO - Processing album artist: 2 Brothers On The 4th Floor
2026-01-23 22:22:17,207 - INFO - Processing album artist: 2 Unlimited
2026-01-23 22:22:17,221 - INFO - Processing album artist: 2Pac
2026-01-23 22:22:17,245 - INFO - Cached 100 files...
2026-01-23 22:22:17,258 - INFO - Processing album artist: 3 Steps Ahead
2026-01-23 22:22:17,259 - INFO - Processing album artist: 4am Kru
2026-01-23 22:22:17,261 - INFO - Processing album artist: 50 Cent
2026-01-23 22:22:17,286 - INFO - Cached 200 files...
2026-01-23 22:22:17,300 - INFO - Processing album artist: 6 Pack (Explicit Version)
2026-01-23 22:22:17,301 - INFO - Processing album artist: 702
2026-01-23 22:22:17,301 - INFO - Processing album artist: 808 State
2026-01-23 22:22:17,303 - INFO - Processi

In [23]:
hhcib_lines = base.read_old_playlist()
matched, unmatched = base.find_matches(hhcib_lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

2026-01-23 22:29:05,492 - INFO - Step 2: Reading playlist: E:\Music\_playlists\How Hard Can It Be.m3u8
2026-01-23 22:29:05,493 - INFO - Read 187 lines from playlist
2026-01-23 22:29:05,493 - INFO - Step 3: Finding matches for playlist entries
2026-01-23 22:29:05,882 - INFO - Matched: 93, Unmatched: 0
2026-01-23 22:29:05,883 - INFO - Step 4: Writing new playlist: E:\Music\_playlists\How Hard Can It Be - Matched.m3u8
2026-01-23 22:29:05,883 - INFO - Wrote 93 entries to new playlist
2026-01-23 22:29:05,884 - INFO - Step 5: Writing unmatched log: E:\Music\_playlists\unmatched.log
2026-01-23 22:29:05,884 - INFO - 
2026-01-23 22:29:05,885 - INFO - SUMMARY
2026-01-23 22:29:05,885 - INFO - ================================================================================
2026-01-23 22:29:05,886 - INFO - Total songs: 93
2026-01-23 22:29:05,886 - INFO - Matched: 93
2026-01-23 22:29:05,886 - INFO - Unmatched: 0
2026-01-23 22:29:05,886 - INFO - Success rate: 100.0%
2026-01-23 22:29:05,887 - INFO - 


In [25]:
base.playlist_path = Path("E:\\Music\\_playlists\\Wedding Party.m3u8")
base.output_path = Path("E:\\Music\\_playlists\\Wedding Party - Matched.m3u8")

hhcib_lines = base.read_old_playlist()
matched, unmatched = base.find_matches(hhcib_lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

2026-01-23 22:31:41,655 - INFO - Step 2: Reading playlist: E:\Music\_playlists\Wedding Party.m3u8
2026-01-23 22:31:41,683 - INFO - Read 555 lines from playlist
2026-01-23 22:31:41,684 - INFO - Step 3: Finding matches for playlist entries
2026-01-23 22:31:42,836 - INFO - Matched: 277, Unmatched: 0
2026-01-23 22:31:42,837 - INFO - Step 4: Writing new playlist: E:\Music\_playlists\Wedding Party - Matched.m3u8
2026-01-23 22:31:42,838 - INFO - Wrote 277 entries to new playlist
2026-01-23 22:31:42,839 - INFO - Step 5: Writing unmatched log: E:\Music\_playlists\unmatched.log
2026-01-23 22:31:42,840 - INFO - 
2026-01-23 22:31:42,840 - INFO - SUMMARY
2026-01-23 22:31:42,841 - INFO - ================================================================================
2026-01-23 22:31:42,841 - INFO - Total songs: 277
2026-01-23 22:31:42,841 - INFO - Matched: 277
2026-01-23 22:31:42,842 - INFO - Unmatched: 0
2026-01-23 22:31:42,842 - INFO - Success rate: 100.0%
2026-01-23 22:31:42,842 - INFO - 
New pl